# latexcodec

latexcodec registers Python codecs that turn Unicode text into LaTeX markup and back (`"é".encode("latex")` gives
`b"\\'e"`), plus the incremental LaTeX lexer behind them. python/pybtex uses it to read the accents and commands inside
BibTeX fields. These cells cover the codec in both directions, its math and target-encoding variants, the lexer, the
incremental (chunked) forms and its error handlers.

In [ ]:
# Example 1: accented names go to LaTeX accent commands and come back unchanged (latexcodec's own use case).
import latexcodec  # importing it registers the "latex" codec family

names = "Gödel, Schrödinger, Erdős"
encoded = names.encode("latex")
print(encoded)
assert encoded == b'G\\"odel, Schr\\"odinger, Erd\\H os', encoded
assert encoded.decode("latex") == names
assert "ç".encode("latex") == b"\\c c" and b"\\c c".decode("latex") == "ç"
print(latexcodec.__name__, "round trip ok")

In [ ]:
# Example 2: math symbols become math mode, and "latex+<encoding>" keeps what the target encoding can hold.
symbols = "α ≤ β".encode("latex")
print(symbols)
assert symbols == b"$\\alpha$ $\\leq$ $\\beta$", symbols

latin1 = "é α".encode("latex+latin1")  # é exists in Latin-1, so it stays a byte; α does not
ascii_only = "é".encode("latex+ascii")   # ASCII has no é, so it becomes the accent command
print(latin1, ascii_only)
assert latin1 == b"\xe9 $\\alpha$", latin1
assert ascii_only == b"\\'e", ascii_only
assert latin1.decode("latex+latin1") == "é α"

In [ ]:
# Example 3: decoding LaTeX markup: accents, ligatures, math and quotes become Unicode; an unknown macro is kept.
source = b"\\'e \\c{c} \\ss{} $\\alpha$ --- ``x''"
text = source.decode("latex")
print(text)
assert text == "é {ç} ß{} α — “x”", text

unknown = b"\\foo{bar}".decode("latex")
print(unknown)
assert unknown == "\\foo {bar}", unknown

In [ ]:
# Example 4: the incremental lexer splits LaTeX into typed tokens (control symbols, math shifts, words).
from latexcodec.lexer import LatexIncrementalLexer, LatexLexer

tokens = list(LatexIncrementalLexer().get_tokens("Schr\\\"odinger's $x^2$", final=True))
print([(t.name, t.text) for t in tokens])
assert ("control_symbol", '\\"') in [(t.name, t.text) for t in tokens]
assert [t.text for t in tokens if t.name == "mathshift"] == ["$", "$"]

raw = [(t.name, t.text) for t in LatexLexer().get_raw_tokens("\\emph{hi} $x$", final=True)]
print(raw)
assert raw[0] == ("control_word", "\\emph"), raw

In [ ]:
# Example 5: chunked decoding waits for a whole command, and the error handlers do what they say.
import codecs

decoder = codecs.getincrementaldecoder("latex")()
parts = [decoder.decode(b'Schr\\"'), decoder.decode(b"odinger"), decoder.decode(b"", final=True)]
print(parts)
assert parts == ["", "Schrö", "dinger"], parts  # the accent command is held back until its letter arrives

try:
    "\u2603".encode("latex")  # a snowman has no LaTeX spelling
except UnicodeEncodeError as error:
    print("strict:", error.reason)
    assert "don't know how to translate" in error.reason
else:
    raise AssertionError("the strict handler accepted a character with no LaTeX spelling")
assert "\u2603".encode("latex", "replace") == b"{\\char9731}"
assert "\u2603".encode("latex", "ignore") == b""